### Project -

In [3]:
# from google.colab import drive
# drive.mount("/content/drive")

# import os
# os.getcwd()

# os.chdir('/content/drive/MyDrive/projects/AIFFEL_quest_eng/NLP/NLP05')


In [5]:
# %reload_ext autoreload
# %autoreload 2

import os
import json
import logging
import copy
from copy import deepcopy
import random
import functools
from typing import Optional, Dict, Sequence, List
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    PreTrainedTokenizerFast,
    GPT2Config,
    GPT2Model,
    pipeline
)

from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

import guide

In [6]:
try:
    root_path = os.path.dirname(os.path.abspath(__file__))
except:
    root_path = os.getcwd()

cfg = {
    "model_name": "skt/kogpt2-base-v2",
    # "device": "cuda" if torch.cuda.is_available() else "cpu",
    "device": torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu"),
    "root_path": root_path,

    "sft_output_dir": root_path + "/test",
    "sft_saved_dir": root_path + "/models/output_1_SFT",
    "sft_num_train_epochs": 1,
    "sft_per_device_train_batch_size": 4,
    "sft_per_device_eval_batch_size": 4,
    "sft_warmup_steps": 5,
    "sft_prediction_loss_only": True,
    "sft_fp16": False # XPU sometimes has issues with fp16 in older versions or specific setups, let's try False or BF16
}

# tokenizer, model 준비
model = AutoModelForCausalLM.from_pretrained(cfg["model_name"]).to(cfg["device"])
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    cfg["model_name"],
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>',
    padding_side="right",
    model_max_length=512,
)

# show_base_model_and_dataset(cfg, model, tokenizer)
guide.show_sft_and_rm_dataset(cfg)
# guide.run_sft(cfg, model, tokenizer)
# guide.run_reward_model(cfg)
# guide.run_ppo(cfg, model, tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/513M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer.json: 0.00B [00:00, ?B/s]


-------------------- show_sft_and_rm_dataset() --------------------
12,000
[{'prompt': '불고기용 고기 한우에요?', 'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.", 'tokens': 193}, {'prompt': '쓰던 앱이 유료로 전환됐어', 'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.", 'tokens': 288}, {'prompt': '여친이랑 다툼', 'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.", 'tokens': 153}]
10,220
[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게

In [10]:
!lm-eval --model hf \
    --model_args pretrained=models/output_1_SFT,tokenizer=skt/kogpt2-base-v2,dtype="float16" \
    --tasks kobest_copa,kobest_hellaswag,kobest_boolq \
    --batch_size auto \
    --limit 500

2026-03-14:11:19:11 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-14:11:19:19 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-14:11:19:21 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-14:11:19:21 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'models/output_1_SFT', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-14:11:19:26 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-14:11:19:29 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 149/149 [00:00<00:00, 153.08it/s, Materializing param=transformer.wte.weight]
The tied weights mapping and config for this model specifies t

In [11]:
!lm-eval --model hf \
    --model_args pretrained=models/output_3_PPO,tokenizer=skt/kogpt2-base-v2,dtype="float16" \
    --tasks kobest_copa,kobest_hellaswag,kobest_boolq \
    --batch_size auto \
    --limit 500

2026-03-14:11:20:04 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-14:11:20:30 INFO     [_cli.run:376] Selected Tasks: ['kobest_copa', 'kobest_hellaswag', 'kobest_boolq']
2026-03-14:11:20:35 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-03-14:11:20:35 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': 'models/output_3_PPO', 'tokenizer': 'skt/kogpt2-base-v2', 'dtype': 'float16'}
2026-03-14:11:20:52 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-03-14:11:20:56 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 149/149 [00:03<00:00, 39.28it/s, Materializing param=transformer.wte.weight]
The tied weights mapping and config for this model specifies to